# Перенос: дообучение на NMT (Пакистан) → проверка на турецком наборе

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-19 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Честная проверка переноса: модель дообучается в одной больнице (NMT, «норма / патология») и проверяется в другой (турецкий набор, «эпилепсия / здоровые»), **не видя турецких данных** — значит, не может выучить их перекосы |
| **Данные** | Обучение: NMT train (Kaggle `mhmdrdwn/nmt-eeg`), 19 каналов, средний референт, 200 Гц. Проверка: Turkish Epilepsy EEG (Kaggle `buraktaci/turkish-epilepsy`), 50 эпилепсия + 71 здоровый, 19 каналов (относительно уха → **пересчитываем к среднему**, как в NMT), 500 → 200 Гц |
| **Модели** | 1) DeepEpilepsy без обучения; 2) DeepEpilepsy, дообученная на NMT; 3) мощности ритмов + лог. регрессия, обученная на NMT (все мощности и только относительные) |
| **Деление** | Обучение — только NMT train (15 % записей — выбор эпохи); проверка — весь турецкий набор, один файл = один человек |
| **Среда** | Kaggle, GPU T4, Internet On; подключить **оба** набора через Add Input |
| **Журнал** | строка в `experiments_log.md`: добавить |

Задачи разные: на NMT — «патология», на турецком — «эпилепсия». У здоровых ЭЭГ должна быть нормальной, но у многих больных эпилепсией рутинная ЭЭГ между приступами тоже нормальна, поэтому очень высоких оценок не ждём.

Связанные блокноты: `h1_2026-10-19_zs_deepepilepsy-nmt.ipynb` (дообучение на NMT, eval 0,83), `h1_2026-10-18_zs_deepepilepsy-turkish.ipynb` (без обучения на турецком, 0,38–0,57).

## 1. Модель

In [5]:
import os, glob, time, subprocess, importlib.util
import numpy as np, pandas as pd, torch, torch.nn.functional as F, mne, scipy.io, scipy.signal as ss
from huggingface_hub import hf_hub_download
from sklearn.metrics import roc_auc_score
mne.set_log_level("ERROR")

REPO = "/kaggle/working/dl_epilepsy_reeg"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://gitlab.com/chum-epilepsy/dl_epilepsy_reeg.git", REPO], check=True)
spec = importlib.util.spec_from_file_location("de_transformers", f"{REPO}/src/epileptology/ml/models/transformers.py")
de = importlib.util.module_from_spec(spec); spec.loader.exec_module(de)

SFREQ, SEG = 200, 30
L = SFREQ * SEG
W_PATH = hf_hub_download("emilelemoine/DeepEpilepsy_v1", "DeepEpilepsy_large_50_30s_augment.pt")
DEV = "cuda" if torch.cuda.is_available() else "cpu"

def new_model():
    m = de.ViT1d(n_class=2, in_channels=19, n_timepoints=L, patch_size=50, tokenizer="conv1d",
                 n_layers=6, n_heads=8, dim_hidden=512, dim_mlp=512, dropout=0.25, attention_dropout=0.25)
    m.load_state_dict(torch.load(W_PATH, map_location="cpu", weights_only=True), strict=True)
    return m.to(DEV)

def scale(x):
    return (x - x.mean(-1, keepdims=True)) / (x.std(-1, keepdims=True) + 1e-20)

def predict(model, Ws):
    """Медиана вероятности по окнам для каждой записи."""
    model.eval(); out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
        for W in Ws:
            p = []
            for b in range(0, len(W), 64):
                x = torch.tensor(scale(W[b:b + 64].astype(np.float32)), dtype=torch.float32).to(DEV)
                p.append(torch.softmax(model(x).float(), 1)[:, 1].cpu().numpy())
            out.append(np.median(np.concatenate(p)))
    return np.array(out)

def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])

BANDS = [(1, 4), (4, 8), (8, 13), (13, 30), (30, 45)]
def band_powers(W):
    fr, P = ss.welch(W, fs=SFREQ, nperseg=400, axis=-1)
    bp = np.stack([P[..., (fr >= a) & (fr < b)].mean(-1) for a, b in BANDS], -1)
    return np.concatenate([np.log(bp + 1e-12).mean(0).ravel(), (bp / bp.sum(-1, keepdims=True)).mean(0).ravel()])

print("устройство:", torch.cuda.get_device_name(0) if DEV == "cuda" else "процессор")

устройство: Tesla T4


## 2. NMT: чтение обучающей части (до 8 окон по 30 с на запись) и проверочной (все окна)

In [8]:
from concurrent.futures import ThreadPoolExecutor

ROOT_NMT = glob.glob("/kaggle/input/**/nmt_scalp_eeg_dataset", recursive=True)[0]
lab = pd.read_csv(f"{ROOT_NMT}/Labels.csv")
paths = {os.path.basename(p): p for p in glob.glob(f"{ROOT_NMT}/**/*.edf", recursive=True)}
lab["path"] = lab["recordname"].map(paths)
lab = lab.dropna(subset=["path"]).reset_index(drop=True)
lab["y"] = (lab["label"].str.lower() == "abnormal").astype(int)

# подвыборка: вся патология + 700 нормальных из train + весь eval
trn = lab[lab["loc"] == "train"]
keep = pd.concat([trn[trn.y == 1], trn[trn.y == 0].sample(700, random_state=0), lab[lab["loc"] == "eval"]])
lab = keep.sort_values("recordname").reset_index(drop=True)
print(pd.crosstab(lab["loc"], lab["label"]))

# параллельная подкачка файлов в кэш (32 потока)
def warm(p):
    with open(p, "rb") as f:
        while f.read(1 << 24):
            pass
t0 = time.time()
with ThreadPoolExecutor(32) as ex:
    for i, _ in enumerate(ex.map(warm, lab.path)):
        if (i + 1) % 100 == 0 or i + 1 == len(lab):
            print(f"подкачано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)

ELECTRODES = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "F7", "F8", "T3", "T4", "T5", "T6", "O1", "O2", "Fz", "Cz", "Pz"]
ALIAS = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}
TARGET = [e.upper() for e in ELECTRODES]
def norm(ch):
    c = ch.upper().replace("EEG", "").strip()
    for s in ["-REF", "-AVG", "-LE", "-AR"]:
        c = c.replace(s, "")
    return ALIAS.get(c.strip(" -_"), c.strip(" -_"))

def load_nmt(path, max_win=None, rng=None):
    raw = mne.io.read_raw_edf(path, preload=True)
    if raw.info["sfreq"] != SFREQ:
        raw.resample(SFREQ)
    names = {norm(c): c for c in raw.ch_names}
    X = raw.get_data(picks=[names[t] for t in TARGET]) * 1e6
    n = X.shape[1] // L
    W = X[:, :n * L].reshape(19, n, L).transpose(1, 0, 2)
    if max_win and n > max_win:
        W = W[np.sort(rng.choice(n, max_win, replace=False))]
    return W

rng = np.random.default_rng(0)
NMT_W, NMT_BP = {}, {}
t0 = time.time()
for i, r in lab.iterrows():
    try:
        W = load_nmt(r.path, max_win=None if r["loc"] == "eval" else 8, rng=rng)
    except Exception:
        continue
    if len(W) == 0:
        continue
    NMT_W[r.recordname] = W.astype(np.float16); NMT_BP[r.recordname] = band_powers(W)
    if (i + 1) % 200 == 0 or i + 1 == len(lab):
        print(f"NMT прочитано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)
lab = lab[lab.recordname.isin(NMT_W)].reset_index(drop=True)
tr, ev = lab[lab["loc"] == "train"].reset_index(drop=True), lab[lab["loc"] == "eval"].reset_index(drop=True)
print(pd.crosstab(lab["loc"], lab["label"]))

label  abnormal  normal
loc                    
eval         90      95
train       325     700
подкачано 100/1210 2 с
подкачано 200/1210 2 с
подкачано 300/1210 8 с
подкачано 400/1210 39 с
подкачано 500/1210 76 с
подкачано 600/1210 109 с
подкачано 700/1210 144 с
подкачано 800/1210 179 с
подкачано 900/1210 214 с
подкачано 1000/1210 228 с
подкачано 1100/1210 235 с
подкачано 1200/1210 242 с
подкачано 1210/1210 242 с
NMT прочитано 200/1210 17 с
NMT прочитано 400/1210 33 с
NMT прочитано 600/1210 52 с
NMT прочитано 800/1210 72 с
NMT прочитано 1000/1210 91 с
NMT прочитано 1200/1210 113 с
NMT прочитано 1210/1210 114 с
label  abnormal  normal
loc                    
eval         90      94
train       323     697


## 3. Турецкий набор: те же 19 каналов в порядке авторов, **средний референт** (как в NMT), 200 Гц, окна 30 с

Куски по 15 с идут подряд (проверено в блокноте `h1_2026-10-18_zs_deepepilepsy-turkish`), поэтому окно 30 с = два соседних куска.

In [9]:
ROOT_TR = glob.glob("/kaggle/input/**/Turkish Epilepsy Dataset", recursive=True)[0]
tfiles = [(f, 1) for f in sorted(glob.glob(f"{ROOT_TR}/epilepsy/*.mat"))] + \
         [(f, 0) for f in sorted(glob.glob(f"{ROOT_TR}/normal/*.mat"))]
from concurrent.futures import ThreadPoolExecutor
t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    for i, _ in enumerate(ex.map(warm, [f for f, _ in tfiles])):
        if (i + 1) % 10 == 0 or i + 1 == len(tfiles):
            print(f"Турция подкачано {i+1}/{len(tfiles)} {time.time()-t0:.0f} с", flush=True)
# индексы турецких каналов в порядке авторов:     Fp1 Fp2 F3 F4 C3 C4 P3 P4 F7  F8  T3  T4  T5  T6  O1  O2  Fz Cz Pz
IDX = [0, 1, 2, 3, 5, 6, 8, 9, 13, 14, 15, 16, 17, 18, 11, 12, 4, 7, 10]

TR_W, TR_BP, y_tr = [], [], []
t0 = time.time()
for i, (f, yy) in enumerate(tfiles):
    d = scipy.io.loadmat(f)["data"]
    X = np.stack([np.stack([d[s, c].ravel() for c in IDX]) for s in range(d.shape[0])])   # (куски, 19, 7500)
    X = X - X.mean(1, keepdims=True)                                                       # средний референт
    X = ss.resample_poly(X, 2, 5, axis=-1)                                                  # 200 Гц, (куски, 19, 3000)
    n = len(X) // 2
    W = np.concatenate([X[0:2*n:2], X[1:2*n:2]], axis=-1)                                  # (окна, 19, 6000)
    TR_W.append(W.astype(np.float16)); TR_BP.append(band_powers(W)); y_tr.append(yy)
    if (i + 1) % 20 == 0 or i + 1 == len(tfiles):
        print(f"Турция {i+1}/{len(tfiles)} {time.time()-t0:.0f} с", flush=True)
y_tr = np.array(y_tr)
print("эпилепсия:", int(y_tr.sum()), "| здоровые:", int((1 - y_tr).sum()))

Турция подкачано 10/121 20 с
Турция подкачано 20/121 31 с
Турция подкачано 30/121 31 с
Турция подкачано 40/121 43 с
Турция подкачано 50/121 53 с
Турция подкачано 60/121 53 с
Турция подкачано 70/121 61 с
Турция подкачано 80/121 66 с
Турция подкачано 90/121 66 с
Турция подкачано 100/121 68 с
Турция подкачано 110/121 72 с
Турция подкачано 120/121 77 с
Турция подкачано 121/121 78 с
Турция 20/121 203 с
Турция 40/121 422 с
Турция 60/121 646 с
Турция 80/121 849 с
Турция 100/121 1121 с
Турция 120/121 1305 с
Турция 121/121 1502 с
эпилепсия: 50 | здоровые: 71


## 4. DeepEpilepsy без обучения на турецком (средний референт)

In [11]:
s0 = predict(new_model(), TR_W)
lo, hi = boot_ci(y_tr, s0)
print(f"Без обучения: ROC AUC = {roc_auc_score(y_tr, s0):.3f} (95% ДИ {lo:.2f}–{hi:.2f}) | "
      f"медиана вероятности: здоровые {np.median(s0[y_tr==0]):.3f}, эпилепсия {np.median(s0[y_tr==1]):.3f}")

Без обучения: ROC AUC = 0.565 (95% ДИ 0.46–0.67) | медиана вероятности: здоровые 0.000, эпилепсия 0.000


## 5. Дообучение DeepEpilepsy на NMT train

Как в блокноте NMT: AdamW, lr 2·10⁻⁵, weight decay 0,05, классы уравновешены, до 8 окон на запись. **3 эпохи** (в прошлый раз лучшей была 1-я), эпоха выбирается по 15 % записей NMT train. Турецкий набор при выборе не используется. Веса сохраняются в `/kaggle/working/deepepilepsy_nmt.pt`.

In [15]:
from sklearn.model_selection import train_test_split

fit_names, val_names = train_test_split(tr.recordname.to_numpy(), test_size=0.15, stratify=tr.y, random_state=0)
ymap = dict(zip(lab.recordname, lab.y))
Xw = np.concatenate([NMT_W[n] for n in fit_names])
yw = np.concatenate([np.full(len(NMT_W[n]), ymap[n]) for n in fit_names])
y_val = np.array([ymap[n] for n in val_names])

EPOCHS, BS = 3, 32
model = new_model()
opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.05)
scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
w = np.where(yw == 1, 0.5 / (yw == 1).sum(), 0.5 / (yw == 0).sum())
best_auc, best_state, best_ep = -1, None, 0
t0 = time.time()
for ep in range(EPOCHS):
    model.train()
    idx = np.random.default_rng(ep).choice(len(yw), size=len(yw), replace=True, p=w / w.sum())
    losses = []
    for b in range(0, len(idx), BS):
        j = idx[b:b + BS]
        x = torch.tensor(scale(Xw[j].astype(np.float32)), dtype=torch.float32).to(DEV)
        t = torch.tensor(yw[j], dtype=torch.long).to(DEV)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
            loss = F.cross_entropy(model(x), t)
        opt.zero_grad(); scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); losses.append(loss.item())
    auc = roc_auc_score(y_val, predict(model, [NMT_W[n] for n in val_names]))
    if auc > best_auc:
        best_auc, best_ep, best_state = auc, ep + 1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"эпоха {ep+1}: потери {np.mean(losses):.3f} | ROC AUC на отложенных записях NMT {auc:.3f} | {time.time()-t0:.0f} с", flush=True)

model.load_state_dict(best_state)
torch.save(best_state, "/kaggle/working/deepepilepsy_nmt.pt")
s_nmt = predict(model, [NMT_W[n] for n in ev.recordname])
print(f"выбрана эпоха {best_ep} | проверка на NMT eval: ROC AUC = {roc_auc_score(ev.y, s_nmt):.3f}")

эпоха 1: потери 0.605 | ROC AUC на отложенных записях NMT 0.719 | 17 с
эпоха 2: потери 0.491 | ROC AUC на отложенных записях NMT 0.728 | 34 с
эпоха 3: потери 0.442 | ROC AUC на отложенных записях NMT 0.754 | 51 с
выбрана эпоха 3 | проверка на NMT eval: ROC AUC = 0.795


## 6. Главное: дообученная на NMT модель → турецкий набор

In [16]:
s_ft = predict(model, TR_W)
lo, hi = boot_ci(y_tr, s_ft)
print(f"DeepEpilepsy, дообученная на NMT: ROC AUC = {roc_auc_score(y_tr, s_ft):.3f} (95% ДИ {lo:.2f}–{hi:.2f}) | "
      f"медиана вероятности: здоровые {np.median(s_ft[y_tr==0]):.3f}, эпилепсия {np.median(s_ft[y_tr==1]):.3f}")

DeepEpilepsy, дообученная на NMT: ROC AUC = 0.663 (95% ДИ 0.56–0.76) | медиана вероятности: здоровые 0.083, эпилепсия 0.390


## 7. Простая модель: мощности ритмов, обучение на NMT train → турецкий набор

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

Xb_nmt = np.stack([NMT_BP[n] for n in tr.recordname]); Xb_tr = np.stack(TR_BP)
half = Xb_nmt.shape[1] // 2
for name, sl in [("все мощности", slice(None)), ("только относительные (без амплитуды)", slice(half, None))]:
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=5000, class_weight="balanced"))
    s_bp = clf.fit(Xb_nmt[:, sl], tr.y).predict_proba(Xb_tr[:, sl])[:, 1]
    lo, hi = boot_ci(y_tr, s_bp)
    print(f"Мощности ритмов, {name:38s}: ROC AUC = {roc_auc_score(y_tr, s_bp):.3f} (95% ДИ {lo:.2f}–{hi:.2f})")

Мощности ритмов, все мощности                          : ROC AUC = 0.782 (95% ДИ 0.69–0.86)
Мощности ритмов, только относительные (без амплитуды)  : ROC AUC = 0.631 (95% ДИ 0.53–0.74)


## 8. Выводы

| Модель (проверка на турецком наборе, не виденном при обучении) | ROC AUC (95% ДИ) | Медиана вероятности: здоровые / эпилепсия |
|---|---|---|
| DeepEpilepsy без обучения | 0,57 (0,46–0,67) | 0,000 / 0,000 |
| **DeepEpilepsy, дообученная на NMT** | **0,66 (0,56–0,76)** | **0,08 / 0,39** |
| Мощности ритмов, обученные на NMT (только относительные, без амплитуды) | 0,63 (0,53–0,74) | — |
| Мощности ритмов, обученные на NMT (все, с амплитудой) | 0,78 (0,69–0,86) | — |

1. **Дообучение на смежном клиническом наборе переносится в другую больницу.** Без обучения модель всем ставила ≈0; после дообучения на NMT вероятности у больных и здоровых разошлись (0,39 против 0,08), ROC AUC вырос с 0,57 до 0,66 — при том, что турецких данных модель не видела, а задача на NMT была другой («патология», а не «эпилепсия»).
2. **Сравнение без амплитуды — наравне с простой моделью**: DeepEpilepsy получает нормированные окна (амплитуду не видит), её честный соперник — относительные мощности ритмов (0,63). Результаты близки.
3. **Все мощности дали 0,78**, но в турецком наборе у больных амплитуда выше (одна амплитуда там даёт 0,79, см. блокнот REVE). Правило «больше мощность — патология», выученное на NMT, совпадает с этим перекосом, поэтому 0,78 нельзя считать чистой заслугой метода.
4. На NMT eval в этом запуске 0,80 (в прошлом — 0,83): обучение шло на подвыборке (1 020 записей вместо 2 225) и всего 3 эпохи, причём качество росло до последней эпохи — модели можно дать больше данных и эпох.
5. **Вывод для MVP:** стратегия «сначала дообучение на смежных клинических данных, потом на целевых» работает — это основа для схемы NMT/TUAB → TUEP → данные клиники.

**Ограничения:** задачи разные (NMT — «патология», Турция — «эпилепсия»); контроль в турецком наборе — здоровые; разная аппаратура и исходный референт (ухо → пересчитан к среднему); на NMT взята подвыборка (вся патология + 700 нормальных), 3 эпохи; гиперпараметры не подбирались.

## 9. Что дальше

- Дообучить на всём NMT и больше эпох; добавить аугментации реальных записей (шум, масштаб амплитуды, сдвиг, отключение каналов) и проверить перенос на турецкий набор снова.
- Ансамбль: дообученная DeepEpilepsy + относительные мощности ритмов.
- Тот же перенос для REVE (дообучение на NMT → турецкий набор) — для честного сравнения моделей.
- Когда придёт доступ к TUH: NMT/TUAB → TUEP.